# Escaping the Morlocks · Lab 02: Apache Iceberg — follow the snapshot graph

> The past is a graph of snapshots, manifests, and files.

## Mission

Replay the exact same incident, capture real Iceberg snapshot IDs, and read each earlier state. Inspect snapshot operations and learn what expiration changes.

## Same incident, same expected states

| State | Mutation | Rows |
|---|---|---:|
| Baseline | Ada + Babbage | 2 |
| Late arrival | Update Babbage; append Larry with an older `event_time` | 3 |
| Correction | Delete Ada | 2 |

Each run creates a random temporary table. No Docker, named production table, remote catalog or Databricks workspace is required for these local OSS labs. Install `requirements.txt`; Spark downloads its pinned connector packages from Maven Central on first run.

## 1 · Start Iceberg 1.9.2 with a local Hadoop catalog

A local warehouse needs no Hive Metastore for native Iceberg. This is deliberately separate from the UniForm Iceberg catalog requirement.

In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
import tempfile
ROOT=Path(tempfile.mkdtemp(prefix="morlocks-iceberg-")).resolve()
ICEBERG="1.9.2"; WAREHOUSE=(ROOT/"warehouse").as_uri()
b=(SparkSession.builder.master("local[2]").appName("Morlocks-Iceberg").config("spark.ui.enabled","false").config("spark.sql.shuffle.partitions","2").config("spark.sql.extensions","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions").config("spark.sql.catalog.morlocks","org.apache.iceberg.spark.SparkCatalog").config("spark.sql.catalog.morlocks.type","hadoop").config("spark.sql.catalog.morlocks.warehouse",WAREHOUSE).config("spark.jars.packages",f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG}") )
spark=b.getOrCreate(); spark.sparkContext.setLogLevel("ERROR")
spark.sql("CREATE NAMESPACE IF NOT EXISTS morlocks.lab"); TABLE="morlocks.lab.events"
print("Spark",spark.version,"Iceberg",ICEBERG,"warehouse",WAREHOUSE)

## 2 · Write baseline, upsert, and delete as separate snapshots

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, shutil, tempfile
from pyspark.sql import functions as F, types as T
PATH=(ROOT/"events").as_uri()
SCHEMA=T.StructType([T.StructField("id",T.LongType(),False),T.StructField("name",T.StringType(),False),T.StructField("segment",T.StringType(),False),T.StructField("balance",T.DoubleType(),False),T.StructField("event_time",T.TimestampType(),False)])
BASE=[(1,"Ada","bronze",100.,datetime(2026,10,24,8,0)),(2,"Babbage","silver",200.,datetime(2026,10,24,8,5))]
LATE=[(2,"Babbage Lovelace","gold",250.,datetime(2026,10,24,8,5)),(99,"Larry Latecomer","time-traveller",42.42,datetime(2026,10,20,9,15))]
def df(rows): return spark.createDataFrame(rows,SCHEMA)
def proof(frame):
 rows=frame.select("id","name","segment","balance",F.date_format("event_time","yyyy-MM-dd HH:mm:ss").alias("event_time")).orderBy("id").collect()
 raw="\n".join("|".join(str(v) for v in r) for r in rows)
 return len(rows),hashlib.sha256(raw.encode()).hexdigest(),rows
df(BASE).writeTo(TABLE).using("iceberg").create()
def snap(): return int(spark.sql(f"SELECT snapshot_id FROM {TABLE}.snapshots ORDER BY committed_at DESC LIMIT 1").first()[0])
s0=snap(); df(LATE).createOrReplaceTempView("incoming")
spark.sql(f"""MERGE INTO {TABLE} t USING incoming s ON t.id=s.id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
s1=snap(); spark.sql(f"DELETE FROM {TABLE} WHERE id=1"); s2=snap()
print("native snapshot IDs:",s0,s1,s2)
spark.sql(f"SELECT snapshot_id,parent_id,committed_at,operation FROM {TABLE}.snapshots ORDER BY committed_at").show(truncate=False)

## 3 · Travel with Iceberg’s native `snapshot-id` option

In [ ]:
def at(s): return spark.read.format("iceberg").option("snapshot-id",s).load(TABLE)
states=[at(s0),at(s1),at(s2)]
for label,state in zip(("baseline","late arrival","correction"),states): print(label,proof(state)[:2])
assert [x.count() for x in states]==[2,3,2]
assert states[0].where("id=99").count()==0 and states[1].where("id=99").count()==1 and states[2].where("id=1").count()==0
print("PASS: the graph resolves all three expected historical states.")

## Conclusions · Iceberg

**Strength:** snapshots and manifests support efficient planning and table features such as branches/tags. Travel by snapshot ID or timestamp.

**Pitfalls:** IDs are opaque, table-specific coordinates, not counters comparable with Delta/Hudi. Snapshot expiration changes which history is queryable; later file cleanup can remove old data. Snapshot ≠ backup.

**Best practice:** save table identity + snapshot ID (or a deliberate tag/branch), and review expiration/retention before maintenance. This lab does not expire snapshots.

In [ ]:
spark.stop(); shutil.rmtree(ROOT,ignore_errors=True); print("PASS: temporary lab cleaned up")